# LatentMind V6 — Colab Runtime

**Agentic analytics** over the `interndb` telecom database: LangGraph state machine,
latent planner (BGE-M3), dual-role SLM with KV-cache hand-off, charts, email drafts, reports.

**Setup order:**
1. Runtime > Change runtime type → **T4 GPU** (or L4)
2. Cell 1 — install deps (~2 min)
3. Cell 2 — mount Drive + clone repo
4. Cell 3 — copy `interndb.sqlite` from Drive
5. Cell 4 — configure + GPU check
6. Cell 5 — load agent
7. Cell 6 — run queries with live token streaming

**Pre-req:** upload `interndb.sqlite` to `MyDrive/LatentDjezzy/interndb.sqlite`

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────
!pip install -q transformers==4.46.3 accelerate tokenizers
!pip install -q sentence-transformers FlagEmbedding
!pip install -q langgraph
!pip install -q bitsandbytes
# Flash Attention 2 — uncomment for 2-4x attention speedup (8 min compile)
# !pip install flash-attn --no-build-isolation
print('\n✓ Dependencies installed')

In [ ]:
# ── Cell 2: Mount Drive + clone repo ─────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, sys
REPO = '/content/latent-djezzy'
if not os.path.exists(REPO):
    !git clone https://github.com/Hamza09Hamza/Latent-Djezzy.git {REPO}
else:
    !git -C {REPO} pull --rebase

if REPO not in sys.path:
    sys.path.insert(0, REPO)

assert os.path.isdir(os.path.join(REPO, 'v6')), 'v6/ missing — push the folder first'
print('\n✓ Repo ready:', sorted(os.listdir(REPO)))

In [ ]:
# ── Cell 3: Copy SQLite DB from Drive ────────────────────────────────────
import shutil, os

SQLITE_SRC = '/content/drive/MyDrive/LatentDjezzy/interndb.sqlite'
SQLITE_DST = '/content/interndb.sqlite'

if os.path.exists(SQLITE_SRC):
    shutil.copy(SQLITE_SRC, SQLITE_DST)
    print(f'✓ DB ready: {os.path.getsize(SQLITE_DST)/1e6:.1f} MB at {SQLITE_DST}')
else:
    print(f'⚠ Not found: {SQLITE_SRC}')
    print('  1. locally: python export_interndb_to_sqlite.py')
    print('  2. upload interndb.sqlite → Drive/LatentDjezzy/')
    print('  3. re-run this cell')

In [ ]:
# ── Cell 4: Configure + GPU check ────────────────────────────────────────
import os, torch

os.environ['V6_USE_SQLITE']   = '1'
os.environ['V6_SQLITE_PATH']  = '/content/interndb.sqlite'
os.environ['V6_SLM_OVERRIDE'] = 'Qwen/Qwen2.5-Coder-3B-Instruct'
os.environ['V6_PLANNER']      = 'prototype'

# Uncomment for 7B model on T4 (halves VRAM via 4-bit NF4):
# os.environ['V6_4BIT'] = '1'
# os.environ['V6_SLM_OVERRIDE'] = 'Qwen/Qwen2.5-Coder-7B-Instruct'

# Uncomment after installing flash-attn in Cell 1:
# os.environ['V6_FLASH_ATTN'] = '1'

if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'✓ GPU : {p.name}')
    print(f'  VRAM: {p.total_memory/1e9:.1f} GB')
    print(f'  CUDA: {torch.version.cuda}')
    print(f'  fp16 support: {p.major >= 7}')   # T4 is sm_75 → True
    print(f'  model: {os.environ["V6_SLM_OVERRIDE"]}')
    print(f'  4-bit: {os.environ.get("V6_4BIT","0")=="1"}')
    print(f'  flash-attn: {os.environ.get("V6_FLASH_ATTN","0")=="1"}')
else:
    print('⚠ No GPU — Runtime > Change runtime type > T4 GPU')

In [ ]:
# ── Cell 5: Load agent ───────────────────────────────────────────────────
# BGE-M3 loads here (~1-2 min). Qwen downloads on first data query (~6 GB).
import time
from v6.graph import LatentMindV6
from v6.state import initial_state

print('Loading V6 (BGE-M3 encoder + latent planner)...')
t0 = time.time()
agent = LatentMindV6()
print(f'✓ Agent ready in {time.time()-t0:.1f}s')

In [ ]:
# ── Cell 6: ask() with live token streaming ───────────────────────────────
import sys, time, torch
from IPython.display import Image, display, Markdown
from v6.state import initial_state
from v6.slm import get_slm

DIV = '─' * 64


def ask(question, thread='colab', stream_sql=True, trace=False):
    """
    Run a query through the full V6 graph.
    SQL generation streams tokens live to the output.
    """
    print(f'\n{DIV}\nQ: {question}\n{DIV}')
    t0 = time.time()

    # ── run the graph (plan → retrieve → route → orchestrate → sql → answer)
    config  = {'configurable': {'thread_id': thread}, 'recursion_limit': 60}
    state_in = initial_state(question, thread)

    # stream node-level events so we see progress live
    intent_shown = False
    result = None
    for event in agent.graph.stream(state_in, config, stream_mode='updates'):
        for node, data in event.items():
            if node == 'plan' and not intent_shown:
                caps = data.get('capabilities', [])
                cap_str = f" + {', '.join(caps)}" if caps else ''
                print(f"[planner → {data.get('intent','?')}{cap_str} | "
                      f"conf: {data.get('confidence',0):.2f}]")
                intent_shown = True
            elif node == 'router':
                tables = data.get('router_tables', [])
                print(f"[router  → tables: {tables}]")
            elif node == 'sql_generate':
                sql = data.get('sql_raw', '')
                if sql and stream_sql:
                    print('\n[SQL]')
                    print(sql)
            elif node == 'sql_execute':
                rows = data.get('rows', [])
                print(f"[exec   → {len(rows)} row(s)]")
            elif node == 'finalize' or node == 'answer':
                result = data

    # if stream didn't capture finalize, invoke normally to get full result
    if result is None or 'answer' not in result:
        result = agent.ask(question, thread_id=thread)

    ans = result.get('answer', '(no answer)')
    print(f'\n{DIV}')
    print(ans)

    # display chart / report if produced
    if result.get('chart_path'):
        display(Image(result['chart_path']))
    if result.get('document_path'):
        try:
            with open(result['document_path']) as f:
                display(Markdown(f.read()))
        except Exception:
            pass

    if trace:
        print('\ntrace:')
        for line in result.get('trace', []):
            print('  ·', line)

    total = time.time() - t0
    t = result.get('timings', {})
    parts = [f'{k.replace("_ms","")} {v/1000:.2f}s'
             for k, v in t.items() if k != 'total_ms' and v]
    print(f'\n⏱  {" + ".join(parts)} = {total:.2f}s total')
    print(DIV)

    # free fragmented GPU memory between queries
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result


print('✓ ask() ready  —  ask("your question")')

## Test queries

Run one at a time. First data query downloads Qwen (~6 GB, cached after).

In [ ]:
# greeting — latent planner must route this away from SQL
ask('hello')

In [ ]:
# definition — answered from knowledge base, no SQL
ask('what does ARPU mean')

In [ ]:
# data — the query that crashed V5 (wilaya column bug)
ask('what was the total revenue in Oran')

In [ ]:
# data — Algiers resolved to 'Alger', both cities in result
ask('compare churn rate between Algiers and Constantine')

In [ ]:
# data + chart
ask('chart the average revenue by wilaya')

In [ ]:
# data + report template
ask('put the average arpu by wilaya in a report')

In [ ]:
# data + email draft (never auto-sends)
ask('email the prepaid churn by wilaya to the finance director')

## Conversation memory

Same `thread_id` keeps context across turns — terse follow-ups work.

In [ ]:
ask('what was the total revenue in Oran', thread='mem-demo')
ask('and what about Constantine?',        thread='mem-demo')

In [ ]:
# full node trace
ask('show the weekly arpu trend for prepaid', trace=True)

In [ ]:
# ── GPU memory status ─────────────────────────────────────────────────────
import torch
if torch.cuda.is_available():
    a   = torch.cuda.memory_allocated()/1e9
    r   = torch.cuda.memory_reserved()/1e9
    tot = torch.cuda.get_device_properties(0).total_memory/1e9
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'  allocated : {a:.1f} GB')
    print(f'  reserved  : {r:.1f} GB')
    print(f'  free      : {tot-r:.1f} GB / {tot:.1f} GB total')

## Optional — stream SQL generation live

Bypass the graph and stream tokens directly from the SLM.

In [ ]:
# Direct streaming from the SLM — see tokens as they're generated
from v6.slm import get_slm

slm = get_slm()
messages = [
    {"role": "system", "content": "You are a SQL expert for an Algerian telecom DB."},
    {"role": "user",   "content": "Write SQL to get total revenue by wilaya using global_revenue joined to dim_location via location_id."}
]

print('Streaming SQL tokens live:')
for token in slm.stream_generate(messages, max_new_tokens=256):
    print(token, end='', flush=True)
print()

## Optional — train the latent planner MLP head

The planner defaults to `prototype` mode (nearest-neighbour, no training).
To train the MLP head for better generalisation on your own query logs:

In [ ]:
!cd /content/latent-djezzy && python3 -m v6.planner_data   # synthesize training data
!cd /content/latent-djezzy && python3 -m v6.train_planner  # train head (~1 min)
# Then set V6_PLANNER=mlp in Cell 4 and re-run from Cell 5